# 跨日期待决市价算法审计
## TL;DR
20260828 与可复现随机日期 20260601：沪市已完成的可比帧全部匹配；历史 ETF 收盘状态校验有误拒；深市股票及 ETF 因市价余量规则中止，不能报告整日匹配率。每市六标的、全天样本，不是全市场验收。

## Context & Methods — Key Assumptions
日期从五类源文件齐全的日期池中以种子 20260906 抽取；标的是人工分层样本，不是随机证券样本。输入为 raw_level2_parquet，不使用 canonical snapshot。严格验证不覆盖 source_dirty 之前旧二进制结果。diagnostic 运行单列，不计正式通过率。
本 notebook 审计已有运行结果和独立源生命周期，不自动重跑全日扫描。重跑入口：analysis/run_cross_date_validation.py（输出目录须换成新的路径）、analysis/run_sz_pending_diagnostic.py、analysis/run_sz_etf_cross_date.py。源证据提取：analysis/inspect_sz_pending_failure.py 与 analysis/inspect_sh_historical_close.py。数量为股／份；价格 Decimal 原样字符串保存；LocalTime 不用于排序或状态判定。

In [1]:
from pathlib import Path
import json, random, hashlib
ROOT = Path.cwd()
if not (ROOT / 'Cargo.toml').exists(): ROOT = ROOT.parent
OUT = ROOT / 'reports/20260906-cross-date-pending'
manifest = json.loads((OUT / 'runs.json').read_text())
assert random.Random(manifest['seed']).choice(manifest['eligible_dates']) == '20260601'
assert manifest['selected_dates'] == ['20260828', '20260601']
print('date pool', len(manifest['eligible_dates']), 'selected', manifest['selected_dates'])
print('binary_sha256', manifest['binary_sha256'])
print('current_binary_same', hashlib.sha256((ROOT/'target/release/qtp-replay').read_bytes()).hexdigest() == manifest['binary_sha256'])


date pool 158 selected ['20260828', '20260601']
binary_sha256 a8acb533acc748ddade117be1d43827ac77c60c8bce65dd642d65a2b9772ef5c
current_binary_same True


In [2]:
runs = [json.loads(p.read_text()) for p in sorted(OUT.glob('*-run.json'))]
for run in runs:
    print(run['date'], run['market'], Path(run['command'][-1]).name, 'exit', run['exit_code'], 'report', bool(run['report']))
matched = errors = excluded = 0
for p in sorted(OUT.glob('*-sh-strict.json')):
    report = json.loads(p.read_text())
    assert report['mismatched'] == report['missing_source'] == 0
    assert sum(v['matched'] for v in report['breakdown'].values()) == report['matched']
    matched += report['matched']; errors += report['data_errors']; excluded += report['excluded_by_status']
    print(p.name, 'matched', report['matched'], 'data_errors', report['data_errors'], 'excluded', report['excluded_by_status'])
assert (matched, errors, excluded) == (61277, 3, 2)
print('SZ aborted runs are NOT zero-mismatch completed validations')


20260601 SH 20260601-sh-strict.json exit 1 report True
20260601 SZ --assume-sz-contiguous-responses exit 1 report False
20260601 SZ 20260601-sz-etfs-strict.json exit 1 report False
20260601 SZ 20260601-sz-strict.json exit 1 report False
20260828 SH 20260828-sh-strict.json exit 0 report True
20260828 SZ --assume-sz-contiguous-responses exit 1 report False
20260828 SZ 20260828-sz-etfs-strict.json exit 1 report False
20260828 SZ 20260828-sz-strict.json exit 1 report False
20260601-sh-strict.json matched 33052 data_errors 3 excluded 1
20260828-sh-strict.json matched 28225 data_errors 0 excluded 1
SZ aborted runs are NOT zero-mismatch completed validations


In [3]:
evidence_files = sorted(OUT.glob('*-order-*.json'))
assert len(evidence_files) == 4
for p in evidence_files:
    case = json.loads(p.read_text())
    life = case['lifecycle']
    assert case['order']['OrderQty'] == case['traded_quantity'] + case['cancelled_quantity'] + case['end_remaining']
    assert all(row['remaining_after'] >= 0 for row in life)
    seqs = [r['ApplSeqNum'] for r in case['unfiltered_channel_context']]
    assert seqs == sorted(set(seqs))
    first = life[0]
    next_row = next(r for r in case['unfiltered_channel_context'] if r['ApplSeqNum'] == first['ApplSeqNum'] + 1)
    print(case['date'], case['symbol'], 'order', case['sequence'], 'quantity', case['order']['OrderQty'], 'trade', case['traded_quantity'], 'cancel', case['cancelled_quantity'], 'remaining', case['end_remaining'], 'prices', case['execution_prices'])
    print('after first response:', next_row['ApplSeqNum'], next_row['SecurityID'])


20260601 159501 order 81834 quantity 213000 trade 213000 cancel 0 remaining 0 prices ['2.1480']
after first response: 81836 159286
20260601 300001 order 1237638 quantity 500 trade 500 cancel 0 remaining 0 prices ['39.3800']
after first response: 1237640 002081
20260828 002475 order 840978 quantity 1300 trade 1300 cancel 0 remaining 0 prices ['57.6000']
after first response: 840980 300263
20260828 159915 order 114537 quantity 14500 trade 14500 cancel 0 remaining 0 prices ['3.4720']
after first response: 114539 159570


In [4]:
phases = json.loads((OUT/'20260601-sh-etf-phases.json').read_text())
for symbol, evidence in phases.items():
    assert evidence['snapshot_status_counts'].get('CCALL', 0) == 0
    assert not any(r['TickBSFlag'].strip() == 'CCALL' for r in evidence['tick_statuses'])
    assert any(r['TickBSFlag'].strip() == 'CLOSE' for r in evidence['tick_statuses'])
    print(symbol, 'first CLOSE', evidence['first_close']['UpdateTime'], 'trades >=14:57', evidence['trades_after_1457'], 'last trade', evidence['last_trade']['TickTime'])


510300 first CLOSE 15:00:03.000 trades >=14:57 1295 last trade 14:59:59.700
513100 first CLOSE 15:00:01.000 trades >=14:57 1651 last trade 14:59:59.830
588000 first CLOSE 15:00:01.000 trades >=14:57 5391 last trade 14:59:59.960


## Takeaways / limitations
沪市修改建议：按市场、品种、交易日选择阶段规则；2026-07-06 前 ETF 允许正常 TRADE→CLOSE，仍按原生逐笔 CLOSE 检查点精确比较，不能删除所有证券的 CCALL 约束。[上交所官方修订说明](https://star.sse.com.cn/aboutus/mediacenter/hotandd/c/c_20260424_10816474.shtml)。
深市修改建议：过滤前保留全通道控制序列；市价余量推断须有已核验的执行结束／子类型契约。源生命周期闭合不证明准确入簿时点；不能将后续被动成交都归并到初始即时成交组，也不能回填过去截面。
本次未改变 Rust 源码、验收规范或匹配窗口。仅报告已执行证据；跨毫秒、空簿 U、真正缺失通道数据等仍需要专门契约／回归测试。深市未完成，不得外推为全市场通过或宣称 source 丢事件。